# **SETUP**

In [21]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

# Load env variables
load_dotenv()

if os.environ.get("GROQ_API_KEY"):
    print("Bro API KEY Variable exists")
else:
    raise ValueError("GROQ_API_KEY not found")

from langchain_core.messages import (
    HumanMessage,
    SystemMessage,
    AIMessage
)

from langchain_core.prompts import ChatPromptTemplate

from langchain_core.output_parsers import (
    StrOutputParser,
    PydanticOutputParser
)

llm_groq = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.7
)

Bro API KEY Variable exists


# **PYDANTIC SCHEMA**

In [22]:
from pydantic import BaseModel
from typing import Literal

class llm_schema(BaseModel):

    movie_summary_flag: Literal[
        "positive",
        "negative"
    ]

llm_structured_output = (
    llm_groq.with_structured_output(llm_schema)
)

# **CHAINS WITH CONDITIONAL CHAINS**

In [23]:
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a movie review evaluator"),
    (
        "human",
        "Please categorize the movie review as positive or negative: {input}"
    )
])

In [24]:
llm_structured_output = (
    llm_groq.with_structured_output(llm_schema)
)

# **CUSTOM RUNNABLE**

In [25]:
from langchain_core.runnables import RunnableLambda

def pydantic_json(input: llm_schema) -> str:

    return input.model_dump()["movie_summary_flag"]

pydantic_json_lambda = RunnableLambda(
    pydantic_json
)

# **CONDITIONAL CHAIN - 1**

In [26]:
linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a LinkedIn post generator"),
    (
        "human",
        "Create a LinkedIn post for the following text: {text}"
    )
])

llm_groq = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.7
)

str_parser = StrOutputParser()

chain_linkedin = (
    linkedin_prompt
    | llm_groq
    | str_parser
)

# **CONDITIONAL CHAIN - 2**

In [27]:
from langchain_core.runnables import (
    RunnableParallel,
    RunnableLambda,
    RunnableBranch
)

In [28]:
def insta_chain(text: dict):

    text = text["text"]

    insta_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an Instagram post generator"),
        (
            "human",
            "Create an Instagram post for the following text: {text}"
        )
    ])

    llm_groq = ChatGroq(
        model="llama-3.3-70b-versatile",
        temperature=0.7
    )

    str_parser = StrOutputParser()

    chain_insta = (
        insta_prompt
        | llm_groq
        | str_parser
    )

    result = chain_insta.invoke({"text": text})

    return result

insta_chain_runnable = RunnableLambda(
    insta_chain
)

# **FINAL ORCHESTRATION**

In [29]:
conditional_chain = RunnableBranch(

    (lambda x: "positive" in x, chain_linkedin),

    insta_chain_runnable
)

final_orchestrator = (
    prompt_template
    | llm_structured_output
    | pydantic_json_lambda
    | conditional_chain
)

In [30]:
final_orchestrator.invoke({
    "input": "I loved this KGF movie"
})

'"Today, I want to take a moment to reflect on the power of positivity. A positive mindset can completely shift our perspective, helping us to tackle challenges with confidence and enthusiasm. It\'s a choice we make every day, and it\'s one that can have a profound impact on our personal and professional lives.\n\nLet\'s spread some positivity today! What are some things that bring a smile to your face and help you stay positive? Share with me in the comments below!\n\n#PositiveVibes #Motivation #Inspiration #MentalHealthMatters #WellnessWednesday"'